# Fase 3 · Búsqueda y selección de hiperparámetros

**Detección de exoplanetas en Kepler con una red multirrama — iteración 2** (más datos, entrenamiento en GPU)

Curso de Deep Learning · Especialización en Inteligencia Artificial · Universidad Nacional de Colombia

**Autores:** Thomas Monnier Granda, Esteban Felipe Genes Noya, Juan Sebastian Higuita Riascos, Alexandra Arias Mazo y Eliel Segura Perez

> **Pregunta de este cuaderno:** ¿las redes del protocolo eran las mejores posibles, y cómo se elige sin hacer trampa?

Este cuaderno no entrena nada: lee las evidencias que dejaron los pasos 1 a 5 (`iteracion_2/*.bat`) y las explica. Cada cifra sale de un archivo del repositorio.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

import apoyo  # iteracion_2/notebooks/apoyo.py: rutas, lectura de evidencias y figuras

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 80)
apoyo.estilo()
print("Raíz del repositorio:", apoyo.RAIZ.name, "| evidencias en", apoyo.EVIDENCIAS.relative_to(apoyo.RAIZ))

## 1. Las reglas de la búsqueda

1. **El espacio se declara antes de entrenar** (`src/temporal/busqueda_it2.py`) y no se cambia al ver resultados.
2. **12 configuraciones por red**, las mismas para todas: la del protocolo (configuración 0) y 11 sorteadas con semilla fija.
3. **Selección anidada:** cada partición elige su configuración mirando solo su validación interna. La prueba nunca decide.
4. **Criterio:** la menor NLL de validación. La NLL (pérdida logarítmica) premia a la vez ordenar bien y dar probabilidades honestas.

Se buscó sobre el tamaño de la red (ancho, bloques extra), la activación, la normalización, la pérdida
(entropía cruzada o con suavizado de etiquetas), la tasa de aprendizaje, la regularización y el tamaño de lote.

In [ ]:
red = "m2_promedio"   # cambia por: m4_escalares, m1_astronet, m2_atencion, m2_gru
cfg = apoyo.configuraciones(red)
columnas = [c for c in cfg.columns if not c.startswith("nll_val_k")]
display(Markdown(f"**{apoyo.REDES[red]}** — las 12 configuraciones (la 00 es el protocolo)"))
display(cfg[columnas].set_index("configuracion"))

## 2. ¿Sirvió buscar?

Para cada red se compara, sobre la prueba, el protocolo fijo con el sistema "con búsqueda" (cada
partición con su configuración elegida). La diferencia es **pareada**: los mismos ejemplos, remuestreados
por estrella 2.000 veces. Una diferencia es *distinguible* si su intervalo del 95 % no contiene el cero.

In [ ]:
display(apoyo.tabla_busqueda())
fig, eje = plt.subplots(figsize=(7, 3))
apoyo.bosque({r: apoyo.resumen_busqueda(r)["auc_prueba"]["diferencia"] for r in apoyo.REDES},
             "Ganancia de AUC por buscar (con búsqueda − protocolo fijo)", referencia=0.0, eje=eje)
apoyo.guardar(fig, "03_ganancia_busqueda")

**Lectura.** Tres de las cinco redes mejoran de forma distinguible, todas por milésimas. La rama 1 y
la GRU se quedan casi siempre con la red del protocolo. La búsqueda ayuda, pero poco.

## 3. ¿Las redes grandes ganan?

Cada punto es una configuración: su tamaño frente a su NLL de validación (menor es mejor).

In [ ]:
fig, ejes = plt.subplots(1, 5, figsize=(15, 2.8), sharey=False)
for eje, (red, nombre) in zip(ejes, apoyo.REDES.items()):
    cfg = apoyo.configuraciones(red)
    eje.scatter(cfg["parametros"], cfg["nll_val_media"], color="#2166ac")
    eje.scatter(cfg.loc[0, "parametros"], cfg.loc[0, "nll_val_media"], color="#b2182b", label="protocolo", zorder=3)
    eje.set_xscale("log"); eje.set_title(nombre, loc="left", fontsize=9); eje.set_xlabel("parámetros")
ejes[0].set_ylabel("NLL de validación"); ejes[0].legend(fontsize=8)
fig.tight_layout(); apoyo.guardar(fig, "03_tamano_vs_nll")

Solo en la rama 2 con promedio gana con claridad una red mayor (el doble de ancha y un bloque más).
En las demás, crecer no garantiza mejorar: las redes más grandes de la rama 1 y de la rama 4 quedan peor.

## 4. Análisis de sensibilidad (exploratorio): ¿y si se eligiera por AUC?

La NLL castiga al suavizado de etiquetas, que baja la confianza a propósito. Para saber si la conclusión
depende del criterio, se repitió la selección usando el AUC de validación. **Este análisis se añadió
después de ver resultados, así que no reemplaza al principal:** solo informa.

In [ ]:
filas = []
for red, nombre in apoyo.REDES.items():
    s = apoyo.sensibilidad(red)
    filas.append({"red": nombre,
                  "AUC eligiendo por NLL": s["auc_prueba"]["seleccion_por_nll"]["estimacion"],
                  "AUC eligiendo por AUC": s["auc_prueba"]["seleccion_por_auc"]["estimacion"],
                  "diferencia de AUC": apoyo.con_intervalo(s["auc_prueba"]["auc_menos_nll"], signo=True),
                  "diferencia de NLL": apoyo.con_intervalo(s["nll_prueba"]["auc_menos_nll"], signo=True),
                  "particiones con la misma elección": f"{s['particiones_con_la_misma_eleccion']} de 5"})
display(pd.DataFrame(filas).set_index("red"))

**Lectura.** No hay un ganador uniforme. Solo en la rama 1 elegir por AUC discrimina mejor, y lo paga
con peor calibración. En las otras cuatro redes, elegir por AUC no mejora el AUC y en varias empeora
la NLL. Se mantiene la NLL como criterio principal; el cuaderno 5 muestra que a nivel de sistema
tampoco conviene cambiarlo.

## 5. Una adaptación que sí se hizo, y por qué es legítima

En la rama 2 la profundidad se limitó a un bloque extra (no dos). Se decidió **antes de entrenar esa
rama**, por coste: una prueba con datos aleatorios midió horas por partición para la combinación mayor.
El motivo no dependió de ningún resultado; queda documentado en el código y en el historial.